---
title: "Classification with LLMs"
pagetitle: "Classification with LLMs"
description-meta: "How to use large language models to categorize records for an investigation, and how to validate results before publication."
description-title: "How to use large language models to categorize records for an investigation, and how to validate results before publication"
author: "Leon Yin"
author-meta: "Leon Yin"
date: "11-06-2026"
bibliography: references.bib
execute:
  enabled: false
keywords: classification, large language models, llm, annotation, validation, inter-rater reliability, codebook
twitter-card:
  title: Classification with LLMs
  description: How to use large language models to categorize records for an investigation, and how to validate the results before you publish.
  image: assets/inspect-element-logo.jpg
open-graph:
  title: Classification with LLMs
  description: How to use large language models to categorize records for an investigation, and how to validate the results before you publish.
  locale: us_EN
  site-name: Inspect Element
  image: assets/inspect-element-logo.jpg
href: classification_llms
---

In [ ]:
#| echo: false
from utils import build_buttons
build_buttons(link='classification_llms',
              github='https://github.com/yinleon/inspect-element/blob/main/classification_llms.ipynb',
              citation=True)

When we hear terms like hate speech, terrorism, or NSFW, you probably have a mental image of what that looks like. 

However, when publishing an article or developing a methodology, precise language is everything -- especially with terminology that our reporting builds upon.

Davey Alba and I have a long-running joke that every story we work on amounts to "annotation journalism," as we always end up labeling and then counting something.

## Annotation Journalism

In the aftermath of the 2024 U.S. Presidential Election, traditional media outlets were dubbing President Trump's win the "podcast election" thanks to another _internet thing_.

To dig in deeper, our reporting team watched hundreds of hours of videos from [nine YouTube podcasters](https://www.bloomberg.com/graphics/2025-youtube-podcast-men-for-trump/) and livestreamers to understand their strategy, messaging, and cultural appeal.

We focused on the types of guests featured -- whether they were comedians, athletes, musicians, or Internet celebrities -- and how often political topics such as voting, war, and transgender identity were discussed.

## Manual labor is a good thing

To do this, we manually extracted which guests appeared in each video by each YouTube show, and created a keyword list for the common themes we encountered. 

A strong starting place for reporting borrows from sociology and ethnography: engaging directly with the source material. In our case it meant watching hours of video over and over again.

Manual groundwork helps reporters identify testable patterns and stumble upon resonant examples that will stick with readers. 

Discussing what you see with your reporting team helps solidify themes and arrive at a logical narrative. Although this kind of work can be unglamorous, making time for these tasks can improve the quality of your work.

::: {.callout-note}
#### What about topic modeling?
In computational social science, [topic modeling](https://guides.library.upenn.edu/penntdm/methods/topic_modeling) is one way to segment bodies of text thematically; however, it is an unsupervised and probabilistic algorithm -- there is no answer key and results can differ between runs. It's worth knowing, but it's often used for exploratory analysis.
:::

## Keywords and expertise

After getting a handle on what these shows and their guests discussed, we consulted subject-matter experts on our guest categories, political topics, and related keywords.

As livestreams can be over 8 hours long, analyzing transcripts with paid AI tools can be expensive and time-consuming. 

Keyword matching is fast and virtually free of charge. It works well for a proof of concept, and the results can be vetted to produce publication-ready findings.

In our investigation, we searched the transcripts for keywords related to the political topics we had identified. We reviewed segments containing each keyword to confirm the topic was discussed and root out false positives.

Most transcription tools return timestamps, so you can use them to generate a link to the exact moment in a video.

<pre>
1. TRANSCRIPT SEGMENT
   {"video_id": "<span style="background-color:#FDE68A; color:#713F12;">To-4jtSzw8U</span>", "start": <span style="background-color:#BFDBFE; color:#1E3A8A;">887.5</span>, "end": <span style="background-color:#BFDBFE; color:#1E3A8A;">888.9</span>,
    "text": "Older people probably will <span style="background-color:#BBF7D0; color:#14532D;">vote</span> for Biden."}
        │
        ▼
2. KEYWORD MATCH   (regex)
   kw_voting = ["<span style="background-color:#BBF7D0; color:#14532D;">vote</span>", "election", "ballot"]  → HIT
        │
        ▼
3. SPREADSHEET OF HITS  with timestamped links
   youtube.com/watch?v=<span style="background-color:#FDE68A; color:#713F12;">To-4jtSzw8U</span>&t=<span style="background-color:#BFDBFE; color:#1E3A8A;">887</span>s
        │
        ▼
4. VERIFY   click the link in the spreadsheet, watch the clip on YouTube
        │
        ▼
5. LABEL
   video_id    | text     | youtube_url | contains_voting? | annotator | comment
   <span style="background-color:#FDE68A; color:#713F12;">To-4jtSzw8U</span> | <span style="background-color:#BBF7D0; color:#14532D;">Older p…</span> | https://…   | <span style="background-color:#E9D5FF; color:#581C87;">TRUE</span>              | LY        |
</pre>

Verification is the name of the game, as we need to be confident that our headline statistic is solid. I use automation (in this case keyword matching) to scale out searching for patterns previously identified through manual review. Confirming also tends to be much faster than trying to label videos from scratch.

::: {.callout-tip}
#### Using Codebooks

Whether you're verifying automated labels or having people annotate data, try to produce a codebook -- which is basically an instruction manual defining categories and how to handle known edge cases. This is a technique from social science used to make annotations more consistent across multiple annotators. 

When labels are subjective they can still differ. [Intercoder reliability](https://rcd.ucsb.edu/node/345) is a metric that you can use to determine how well your annotators agree on labels.
:::


The same principle can be applied to other forms of automation such as using a language model or other predictive technology to produce the same categorization system.

## When to use LLMs?

You may be wondering: why not throw a language model at the problem and call it a day? 

LLMs are not a hammer to throw at every problem. They are neither magic nor sci-fi "[enhance](https://knowyourmeme.com/memes/zoom-and-enhance)" functions. Using LLMs, AI, or any technical tool for that matter is about solving problems _reliably_.

For example, categorizing something as "political" is not straightforward. Besides elected officials and elections, what constitutes political? 

::: {.callout-tip}
#### Pro tip: Ask narrow questions
"Tell me if this is political" is too blunt. Concrete categories like "elections and voting," "immigration," or "war" yield more precise results, and avoid the assumptions that loaded terms carry.
:::

As AI assistants make most coding tasks trivial, our strength as investigators lies in the questions we ask. How do we assess vulnerabilities in our logic? Are our questions driven by a hypothesis? What assumptions are left unsaid, and how could they mislead readers? What are edge cases that undermine the hypothesis or break the system we designed?

For now, let's look at an example of classifying text using a local language model and verifying the results in an annotation interface.

# Tutorial

1. Download Ollama
2. Preview the text and glean some categories.
3. Run subset text through Ollama with Pydantic. One category = one API call.
4. Verify the text using a coding interface. Marking things as "?" and interesting
5. Intercoder reliability and golden set
6. Make changes to prompts etc
7. Run on everything

# Setup

Install [Ollama](https://ollama.com) to run models locally, then download a model:

```bash
brew install ollama
ollama serve
ollama pull qwen3.5:4b
```

Install the Python packages: the [Ollama Python library](https://github.com/ollama/ollama-python), [Pydantic](https://docs.pydantic.dev), and [pandas](https://pandas.pydata.org).

```bash
pip install ollama pydantic pandas
```

In [47]:
import os
import glob
import json
import gzip

import pandas as pd
from tqdm import tqdm
from ollama import chat
from pydantic import BaseModel


Here we will work with a sample of transcripts taken from YouTube stored in Whisper's JSON format (a common AI transcription tool).

It is a subset of videos from one podcast.

In [36]:
fn_transcripts = "data/input/classification/full_send_segments_sample.json.gz"

transcripts = json.load(
    gzip.open(fn_transcripts, "rt")
)

df = pd.json_normalize(
    transcripts, record_path="segments", meta=["video_id"]
)

Here's a simple keyword list search:

In [54]:
kw_voting = [
    'vote', 'election', 'ballot'
]
df['kw_voting'] = df['text'].str.contains(
    '|'.join(kw_voting), regex=True, case=False
)

Pydantic allows us to define a data schema for the language model's output. This is way better than what I used to do by defining this within a prompt, as we can use Ollama's `format` parameter directly (read the [documentation](https://docs.ollama.com/capabilities/structured-outputs#python-2)).

In [45]:
class VotingLabel(BaseModel):
    """
    This defines the data schema of the LLM response
    """
    mentions_voting: bool | None
    quote: str

Now we will feed each transcript segment through the local language model and compile the labels.

In [49]:
dir_labels = "data/intermediary/voting_labels/"
os.makedirs(dir_labels, exist_ok=True)

for i, row in tqdm(df.iterrows(), total=len(df)):
    fn_out = dir_labels + f"{row['video_id']}-{row['id']}.json"
    
    if not os.path.exists(fn_out):
        prompt = f"Does this podcast transcript mention elections or voting?\n\n{row.text}"          
        response = chat(
            model="qwen3.5:4b",
            messages=[{
                "role": "user", 
                "content": prompt
            }],
            format=VotingLabel.model_json_schema(),
            think=False,
        )
        
        label = VotingLabel.model_validate_json(
            response.message.content
        )
            
        with open(fn_out, "w") as f:
            f.write(label.model_dump_json())

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [05:37<00:00,  1.69s/it]


In [50]:
labels = []
for fn in glob.glob(dir_labels + "*.json"):
    video_id, segment_id = fn.split("/")[-1].split(".json")[0].rsplit("-", 1)
    with open(fn) as f:
        label = json.load(f)
    labels.append({"video_id": video_id, "id": int(segment_id), **label})

df_labels = pd.DataFrame(labels)

In [52]:
df_labels['mentions_voting'].value_counts()

mentions_voting
False    109
True      91
Name: count, dtype: int64

In [55]:
# line up keyword hits and llm labels by segment
df_merged = df.merge(df_labels, on=["video_id", "id"], how="left")

pd.crosstab(
    df_merged["kw_voting"],
    df_merged["mentions_voting"],
    margins=True,
)

mentions_voting,False,True,All
kw_voting,,,
False,52,53,105
True,57,38,95
All,109,91,200


## Data Annotations

Verification is necessary prior to publication; you shouldn't let an automated classification rip and publish the results. If you want to evaluate multiple approaches to producing classifications, hand-labeling a "golden set" to benchmark performance is an approach I often take. It produces an estimate not only of how well the automated approach is going to work but also the annotation rate for how long a larger task will take.

Here's an example of a data annotation interface with basic functionality. In my experience, hot-key mapping and single verification tasks help reduce the time needed to verify labels.

<iframe id="label-segments" src="assets/label-segments.html" width="100%" height="300" style="border:0;"></iframe>
<script>
  // resize the labeling tool's frame to fit its content
  window.addEventListener("message", (e) => {
    const frame = document.getElementById("label-segments");
    if (e.source === frame.contentWindow && e.data.labelSegmentsHeight) {
      frame.style.height = e.data.labelSegmentsHeight + "px";
    }
  });
</script>

The "interesting" flag saves anecdotes to review and use in publication, whereas "skip" can be used to mark edge cases to discuss with annotators.

The coding interface should be super simple by default, but provide the ability to drill in further for context. In this case you can actually watch the time-stamped clip. For text, for example, you could show an abbreviated segment and allow for expanding to the full text or more metadata.

When running such tasks, I tend to use a simple command-line implementation built on Python (the user interface above is purely demonstrative). These are fairly straightforward to vibe-code, and if your reporting team has a coding assistant installed you can just send them the codebase and dataset and have it set up easily. 

Here's the Python script for reference.

::: {.callout-tip}
#### Pro tip: Information Security
Keeping this task entirely local -- including the classification itself -- ensures data does not get read by private AI firms and other tech platforms. This is a requirement if the data you're working with is sensitive.
:::



# Citation

To cite this chapter, please use the following BibTeX entry:

<pre>
@incollection{yin2026classification,
  author    = {Yin, Leon},
  title     = {Classification with LLMs},
  booktitle = {Inspect Element: A practical guide to data investigations},
  year      = {2026},
  editor    = {Yin, Leon and Sapiezynski, Piotr},
  note      = {\url{https://inspectelement.org}}
}
</pre>